# E-commerce Fulfilment & Delivery Performance Analysis

## Notebook 02: Data Cleaning and Preparation

This notebook focuses on cleaning and preparing the raw Olist e-commerce data for analysis.

The purpose of this notebook is to:

- Load the raw CSV files
- Convert important date columns into proper datetime format
- Focus on delivered orders for delivery performance analysis
- Create delivery-related calculated fields
- Prepare clean datasets for later SQL, Power BI and exploratory analysis

The first notebook focused on understanding the raw data. This notebook starts preparing the data for analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Define project folders

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

# Check that both folders exist
print("Raw data folder exists:", raw_data_path.exists())
print("Processed data folder exists:", processed_data_path.exists())

## Load Raw Data

In this section, I will load the raw CSV files again.

Although the files were already loaded in Notebook 01, each notebook should be able to run independently. This means Notebook 02 should not depend on Notebook 01 being open or already run.

In [ ]:
# Load raw datasets

customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_data_path / "product_category_name_translation.csv")

print("Raw datasets loaded successfully.")

## Initial Orders Table Check

The orders table is the main table for delivery performance analysis because it contains:

- Order ID
- Customer ID
- Order status
- Purchase timestamp
- Carrier delivery date
- Customer delivery date
- Estimated delivery date

Before cleaning, I will inspect the structure of the orders table.

In [ ]:
# Preview the orders table

orders.head()

In [ ]:
# Check columns and data types in the orders table

orders.info()

## Convert Date Columns

The raw orders table contains several date columns.

At first, pandas reads these columns as text. To calculate delivery days and late delivery, these columns need to be converted into datetime format.

Datetime format allows Python to calculate the difference between dates.

In [ ]:
# Create a copy of the orders table for cleaning

orders_clean = orders.copy()

# List of date columns to convert
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert date columns to datetime format
for column in date_columns:
    orders_clean[column] = pd.to_datetime(orders_clean[column], errors="coerce")

# Check data types after conversion
orders_clean[date_columns].dtypes

## Filter to Delivered Orders

For delivery performance analysis, I will focus on orders with an order status of delivered.

This is because late delivery can only be measured properly when the actual customer delivery date is available.

Cancelled, unavailable, shipped or invoiced orders may be useful for other analysis later, but they are not the main focus for delivery performance KPIs.

In [ ]:
# Check order status counts

orders_clean["order_status"].value_counts()

In [ ]:
# Filter to delivered orders only

delivered_orders = orders_clean[orders_clean["order_status"] == "delivered"].copy()

# Check the number of delivered orders
print("Total orders:", len(orders_clean))
print("Delivered orders:", len(delivered_orders))

## Create Delivery Performance Fields

In this section, I will create new calculated fields for delivery analysis.

These fields will help answer key business questions such as:

- How long did orders take to reach customers?
- Were orders delivered before or after the estimated delivery date?
- How many days late were late orders?
- Which orders were delivered late?

In [ ]:
# Create delivery performance fields

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["estimated_delivery_days"] = (
    delivered_orders["order_estimated_delivery_date"] 
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"] 
    - delivered_orders["order_estimated_delivery_date"]
).dt.days

delivered_orders["is_late_delivery"] = delivered_orders["delay_days"] > 0

# Preview the new fields
delivered_orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delay_days",
        "is_late_delivery"
    ]
].head()

In [ ]:
# Summary statistics for delivery performance fields

delivered_orders[
    [
        "delivery_days",
        "estimated_delivery_days",
        "delay_days"
    ]
].describe()

In [ ]:
# Late delivery count and rate

late_delivery_count = delivered_orders["is_late_delivery"].sum()
delivered_order_count = len(delivered_orders)
late_delivery_rate = late_delivery_count / delivered_order_count

print("Delivered orders:", delivered_order_count)
print("Late deliveries:", late_delivery_count)
print("Late delivery rate:", round(late_delivery_rate * 100, 2), "%")

## Export Cleaned Orders Dataset

The cleaned delivered orders dataset will be saved into the processed data folder.

This processed file will be used later for SQL analysis, Power BI dashboards and further Python analysis.

In [ ]:
# Export cleaned delivered orders dataset

output_file = processed_data_path / "cleaned_delivered_orders.csv"

delivered_orders.to_csv(output_file, index=False)

print("Cleaned delivered orders file saved to:")
print(output_file)

In [ ]:
# Check that the processed file was created

output_file.exists()

## Create Analysis-Ready Master Dataset

The cleaned delivered orders table contains the delivery performance fields, but most business analysis requires data from multiple tables.

In this section, I will create one analysis-ready dataset by combining:

- delivered orders
- customer location
- order item and freight information
- payment information
- review score
- product category information
- seller location information

Some tables contain more than one row per order, so I will aggregate them to order level before joining. This avoids accidentally duplicating orders during the merge.

In [ ]:
# Prepare customer location data

customers_clean = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

customers_clean.head()

In [ ]:
# Aggregate order item data to order level

order_items_summary = (
    order_items
    .groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        total_item_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean")
    )
    .reset_index()
)

order_items_summary.head()

In [ ]:
# Aggregate payment data to order level

payments_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_summary.head()

In [ ]:
# Aggregate review data to order level

reviews_summary = (
    order_reviews
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
)

reviews_summary.head()

In [ ]:
# Join order items to product and seller details

order_items_enriched = (
    order_items
    .merge(products, on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(sellers, on="seller_id", how="left")
)

order_items_enriched.head()

In [ ]:
# Create product and seller summary at order level

product_seller_summary = (
    order_items_enriched
    .groupby("order_id")
    .agg(
        main_product_category=("product_category_name_english", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        main_seller_state=("seller_state", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_length_cm=("product_length_cm", "mean"),
        avg_product_height_cm=("product_height_cm", "mean"),
        avg_product_width_cm=("product_width_cm", "mean")
    )
    .reset_index()
)

product_seller_summary.head()

In [ ]:
# Create the analysis-ready master dataset

master_orders = (
    delivered_orders
    .merge(customers_clean, on="customer_id", how="left")
    .merge(order_items_summary, on="order_id", how="left")
    .merge(payments_summary, on="order_id", how="left")
    .merge(reviews_summary, on="order_id", how="left")
    .merge(product_seller_summary, on="order_id", how="left")
)

master_orders.head()

In [ ]:
# Check the size of the master dataset

print("Delivered orders rows:", len(delivered_orders))
print("Master dataset rows:", len(master_orders))
print("Master dataset columns:", master_orders.shape[1])

In [ ]:
# Check missing values in the master dataset

master_missing = (
    master_orders
    .isna()
    .sum()
    .sort_values(ascending=False)
)

master_missing[master_missing > 0]

In [ ]:
# Export the analysis-ready master dataset

master_output_file = processed_data_path / "analysis_ready_orders.csv"

master_orders.to_csv(master_output_file, index=False)

print("Analysis-ready dataset saved to:")
print(master_output_file)

In [ ]:
master_output_file.exists()